In [ ]:
# ============================================================
#  SHADOW MASKING PIPELINE
# ============================================================

import os
import cv2
import glob
import shutil
import numpy as np
import pandas as pd
from google.colab import drive

# 1. SETUP & MOUNT
drive.mount('/content/drive', force_remount=True)

REPO_DIR = "/content/ShadowDetection2021"
IMG_DIR = "/content/drive/My Drive/Apple/RGB"
OUT_DIR = "/content/drive/My Drive/Apple/mask"
VIS_DIR = "/content/drive/My Drive/Monkey/renders/centroid_visuals"
SHADOW_CLEAN_DIR = "/content/shadow_clean"

WEIGHTS_PATH = f"{REPO_DIR}/models/SBU_model.pth"

# --- CLEAR OLD DATA ---
for d in [OUT_DIR, VIS_DIR, SHADOW_CLEAN_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# 2. CLONE OR RESET REPO
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/guanhuankang/ShadowDetection2021.git
else:
    # CRITICAL: Clean up the broken syntax from previous runs
    %cd {REPO_DIR}
    !git reset --hard HEAD
    !git clean -fd

# 3. DOWNLOAD WEIGHTS
os.makedirs(f"{REPO_DIR}/models", exist_ok=True)
if not os.path.exists(WEIGHTS_PATH):
    !gdown "https://drive.google.com/uc?id=1TY8O5F9GPB0Zv7CxQoAUaqQTbJFLdb6a" -O {WEIGHTS_PATH}

# 4. DISABLE DENSECRF (Hotfix for misc.py)
misc_code = """
import os
import torch
def check_mkdir(d):
    os.makedirs(d, exist_ok=True)
def loadModel(model, path):
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model
def crf_refine(img, annos):
    return annos
"""
with open(f"{REPO_DIR}/misc.py", "w") as f:
    f.write(misc_code)

# 5. CREATE CONFIG
with open(f"{REPO_DIR}/config.py", "w") as f:
    f.write(f'img_path = "{IMG_DIR}"\nout_path = "{OUT_DIR}"\ngt_path = "{OUT_DIR}"\nmodel = "{WEIGHTS_PATH}"\n')

# 5.5 SAFE INFER.PY REWRITE (Removes the calcBER block completely)
if os.path.exists(f"{REPO_DIR}/infer.py"):
    with open(f"{REPO_DIR}/infer.py", "r") as f:
        lines = f.readlines()

    clean_lines = []
    skip_mode = False

    for line in lines:
        # Start dropping lines once we hit the calcBER function definition
        if "def calcBER():" in line:
            skip_mode = True
            continue
        # Stop dropping lines when we hit the main execution guard
        if 'if __name__ == "__main__":' in line:
            skip_mode = False

        if not skip_mode:
            # Replace the actual execution of calcBER() with a safe pass statement
            if "calcBER()" in line and not line.strip().startswith("#"):
                line = line.replace("calcBER()", "pass")
            clean_lines.append(line)

    with open(f"{REPO_DIR}/infer.py", "w") as f:
        f.writelines(clean_lines)

# 6. RUN INFERENCE
%cd {REPO_DIR}
!python infer.py

# 7. PROCESS MASKS + CENTROIDS
mask_files = sorted(glob.glob(f"{OUT_DIR}/*.png"))
print(f"\n📁 New masks generated by model: {len(mask_files)}")

csv_rows = []
MIN_AREA = 50

for path in mask_files:
    name = os.path.basename(path)
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None: continue

    _, binary = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []

    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_AREA: continue
        x, y, w, h = cv2.boundingRect(c)
        valid.append((y + h, area, c))

    if not valid: continue

    valid.sort(key=lambda x: x[0], reverse=True)
    _, area, shadow = valid[0]

    clean = np.zeros_like(binary)
    cv2.drawContours(clean, [shadow], -1, 255, -1)
    cv2.imwrite(f"{SHADOW_CLEAN_DIR}/{name}", clean)

    M = cv2.moments(shadow)
    cx = int(M["m10"]/M["m00"]) if M["m00"] != 0 else -1
    cy = int(M["m01"]/M["m00"]) if M["m00"] != 0 else -1
    csv_rows.append([name, cx, cy, area])

    vis = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)
    if cx != -1:
        cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)
    cv2.imwrite(f"{VIS_DIR}/{name}", vis)

# 8. SAVE CSV
csv_path = "/content/drive/My Drive/Monkey/renders/shadow_centroids_updated.csv"
df = pd.DataFrame(csv_rows, columns=["filename", "centroid_x", "centroid_y", "area_pixels"])
df.to_csv(csv_path, index=False)

print("\n✅ RE-RUN COMPLETE")
print(f"Total processed and saved to CSV: {len(df)}")

Mounted at /content/drive
/content/ShadowDetection2021
HEAD is now at 16408cb raw training scripts
Removing models/SBU_model.pth
Downloading...
From (original): https://drive.google.com/uc?id=1TY8O5F9GPB0Zv7CxQoAUaqQTbJFLdb6a
From (redirected): https://drive.google.com/uc?id=1TY8O5F9GPB0Zv7CxQoAUaqQTbJFLdb6a&confirm=t&uuid=a31d9007-d5d4-4a25-a6b9-104a175c50dd
To: /content/ShadowDetection2021/models/SBU_model.pth
100% 412M/412M [00:02<00:00, 140MB/s]
/content/ShadowDetection2021
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt101_32X8D_Weights.IMAGENET1K_